# Apache Spark com Apache Iceberg

Demonstração de operações CRUD (INSERT, UPDATE, DELETE) com **PySpark** e **Apache Iceberg**.

**Cenário:** Sistema de Gestão de Vendas — TechStore  
**Tabelas:** local.vendas.clientes, local.vendas.produtos, local.vendas.pedidos

In [1]:
import subprocess, os

# Limpa warehouse anterior usando rm -rf (confiavel no WSL/OneDrive)
for d in ['/tmp/spark-wh-iceberg', os.path.expanduser('~/metastore_db')]:
    r = subprocess.run(['rm', '-rf', d], capture_output=True, text=True)
    status = 'removido' if r.returncode == 0 else f'erro: {r.stderr.strip()}'
    print(f'{d} — {status}')


/tmp/spark-wh-iceberg — removido
/home/thiago/metastore_db — removido


In [2]:
from pyspark.sql import SparkSession
import logging

logging.getLogger('py4j').setLevel(logging.WARNING)


In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('IcebergLocalDev')
    .config('spark.jars.packages',
            'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1')
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local',
            'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', '/tmp/spark-wh-iceberg')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark


26/05/07 00:34:33 WARN Utils: Your hostname, LAPTOP-NGVDQKNB resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/07 00:34:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/mazuc/OneDrive/%c3%81rea%20de%20Trabalho/SATC/5%c2%b0%20FASE/Engenharia%20de%20Dados/Trabalho%20Apache%20spark%20iceberg%20e%20Data%20Lake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thiago/.ivy2/cache
The jars for the packages stored in: /home/thiago/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ecb24269-f0b7-4110-a40b-c058a656eeba;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1!iceberg-spark-runtime-3.5_2.12.jar (1564ms)
:: resolution report :: resolve 882ms :: artifacts dl 1568ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted

## Cenário — TechStore

Mesmo sistema de vendas, agora com **Apache Iceberg**.

### Modelo ER
```
CLIENTES (1) ----< PEDIDOS >---- (N) PRODUTOS
```

## DDL — Criação do Namespace e Tabelas

In [4]:
# Cria namespace (equivalente a schema/database)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.vendas")
spark.sql("SHOW NAMESPACES IN local").show()


+---------+
|namespace|
+---------+
|   vendas|
+---------+



In [5]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.vendas.clientes (
        id      INT,
        nome    STRING,
        email   STRING,
        cidade  STRING,
        estado  STRING
    ) USING iceberg
""")
spark.sql("SELECT * FROM local.vendas.clientes").show()


+---+----+-----+------+------+
| id|nome|email|cidade|estado|
+---+----+-----+------+------+
+---+----+-----+------+------+



In [6]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.vendas.produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     FLOAT,
        estoque   INT
    ) USING iceberg
""")
spark.sql("SELECT * FROM local.vendas.produtos").show()


+---+----+---------+-----+-------+
| id|nome|categoria|preco|estoque|
+---+----+---------+-----+-------+
+---+----+---------+-----+-------+



In [7]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.vendas.pedidos (
        id          INT,
        cliente_id  INT,
        produto_id  INT,
        quantidade  INT,
        data_pedido STRING,
        status      STRING
    ) USING iceberg
""")
spark.sql("SELECT * FROM local.vendas.pedidos").show()


+---+----------+----------+----------+-----------+------+
| id|cliente_id|produto_id|quantidade|data_pedido|status|
+---+----------+----------+----------+-----------+------+
+---+----------+----------+----------+-----------+------+



## INSERT — Inserindo Dados

In [8]:
spark.sql("""
    INSERT INTO local.vendas.clientes VALUES
        (1, 'Ana Silva',       'ana@email.com',      'Sao Paulo',      'SP'),
        (2, 'Carlos Oliveira', 'carlos@email.com',   'Rio de Janeiro', 'RJ'),
        (3, 'Maria Santos',    'maria@email.com',    'Curitiba',       'PR'),
        (4, 'Joao Costa',      'joao@email.com',     'Porto Alegre',   'RS'),
        (5, 'Fernanda Lima',   'fernanda@email.com', 'Belo Horizonte', 'MG')
""")
spark.sql("SELECT * FROM local.vendas.clientes").show()


[Stage 1:>                                                          (0 + 1) / 1]

+---+---------------+------------------+--------------+------+
| id|           nome|             email|        cidade|estado|
+---+---------------+------------------+--------------+------+
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|
+---+---------------+------------------+--------------+------+



In [9]:
spark.sql("""
    INSERT INTO local.vendas.produtos VALUES
        (1, 'Notebook Dell',      'Informatica',  3599.99, 15),
        (2, 'Smartphone Samsung', 'Celulares',    1299.00, 50),
        (3, 'Monitor LG 27',      'Informatica',   899.90, 30),
        (4, 'Teclado Mecanico',   'Perifericos',   349.90, 100),
        (5, 'Mouse Logitech',     'Perifericos',   159.90, 80)
""")
spark.sql("SELECT * FROM local.vendas.produtos").show()


+---+------------------+-----------+-------+-------+
| id|              nome|  categoria|  preco|estoque|
+---+------------------+-----------+-------+-------+
|  1|     Notebook Dell|Informatica|3599.99|     15|
|  2|Smartphone Samsung|  Celulares| 1299.0|     50|
|  3|     Monitor LG 27|Informatica|  899.9|     30|
|  4|  Teclado Mecanico|Perifericos|  349.9|    100|
|  5|    Mouse Logitech|Perifericos|  159.9|     80|
+---+------------------+-----------+-------+-------+



In [10]:
spark.sql("""
    INSERT INTO local.vendas.pedidos VALUES
        (1, 1, 2, 2, '2024-01-10', 'entregue'),
        (2, 2, 1, 1, '2024-01-12', 'entregue'),
        (3, 3, 3, 1, '2024-01-15', 'em_transporte'),
        (4, 1, 4, 1, '2024-01-20', 'processando'),
        (5, 4, 5, 3, '2024-01-22', 'cancelado')
""")
spark.sql("SELECT * FROM local.vendas.pedidos").show()


+---+----------+----------+----------+-----------+-------------+
| id|cliente_id|produto_id|quantidade|data_pedido|       status|
+---+----------+----------+----------+-----------+-------------+
|  1|         1|         2|         2| 2024-01-10|     entregue|
|  2|         2|         1|         1| 2024-01-12|     entregue|
|  3|         3|         3|         1| 2024-01-15|em_transporte|
|  4|         1|         4|         1| 2024-01-20|  processando|
|  5|         4|         5|         3| 2024-01-22|    cancelado|
+---+----------+----------+----------+-----------+-------------+



## Consulta com JOIN

In [11]:
spark.sql("""
    SELECT
        p.id         AS pedido_id,
        c.nome       AS cliente,
        pr.nome      AS produto,
        p.quantidade,
        p.status,
        p.data_pedido
    FROM local.vendas.pedidos p
    JOIN local.vendas.clientes c  ON p.cliente_id = c.id
    JOIN local.vendas.produtos pr ON p.produto_id = pr.id
    ORDER BY p.id
""").show(truncate=False)


+---------+---------------+------------------+----------+-------------+-----------+
|pedido_id|cliente        |produto           |quantidade|status       |data_pedido|
+---------+---------------+------------------+----------+-------------+-----------+
|1        |Ana Silva      |Smartphone Samsung|2         |entregue     |2024-01-10 |
|2        |Carlos Oliveira|Notebook Dell     |1         |entregue     |2024-01-12 |
|3        |Maria Santos   |Monitor LG 27     |1         |em_transporte|2024-01-15 |
|4        |Ana Silva      |Teclado Mecanico  |1         |processando  |2024-01-20 |
|5        |Joao Costa     |Mouse Logitech    |3         |cancelado    |2024-01-22 |
+---------+---------------+------------------+----------+-------------+-----------+



## UPDATE — Atualizando Dados

In [12]:
spark.sql("UPDATE local.vendas.pedidos SET status = 'entregue' WHERE id = 3")
spark.sql("SELECT * FROM local.vendas.pedidos WHERE id = 3").show()


+---+----------+----------+----------+-----------+--------+
| id|cliente_id|produto_id|quantidade|data_pedido|  status|
+---+----------+----------+----------+-----------+--------+
|  3|         3|         3|         1| 2024-01-15|entregue|
+---+----------+----------+----------+-----------+--------+



In [13]:
spark.sql("UPDATE local.vendas.produtos SET preco = 1199.00, estoque = 45 WHERE id = 2")
spark.sql("SELECT * FROM local.vendas.produtos WHERE id = 2").show()


+---+------------------+---------+------+-------+
| id|              nome|categoria| preco|estoque|
+---+------------------+---------+------+-------+
|  2|Smartphone Samsung|Celulares|1199.0|     45|
+---+------------------+---------+------+-------+



## DELETE — Removendo Dados

In [14]:
spark.sql("DELETE FROM local.vendas.pedidos WHERE status = 'cancelado'")
spark.sql("SELECT * FROM local.vendas.pedidos").show()


+---+----------+----------+----------+-----------+-----------+
| id|cliente_id|produto_id|quantidade|data_pedido|     status|
+---+----------+----------+----------+-----------+-----------+
|  1|         1|         2|         2| 2024-01-10|   entregue|
|  2|         2|         1|         1| 2024-01-12|   entregue|
|  4|         1|         4|         1| 2024-01-20|processando|
|  3|         3|         3|         1| 2024-01-15|   entregue|
+---+----------+----------+----------+-----------+-----------+



## ALTER TABLE — Evolução de Schema

O Iceberg suporta evolução de schema sem recriar a tabela.

In [15]:
spark.sql("ALTER TABLE local.vendas.clientes ADD COLUMN telefone STRING")
spark.sql("SELECT * FROM local.vendas.clientes").show()


+---+---------------+------------------+--------------+------+--------+
| id|           nome|             email|        cidade|estado|telefone|
+---+---------------+------------------+--------------+------+--------+
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|    NULL|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|    NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|    NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|    NULL|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|    NULL|
+---+---------------+------------------+--------------+------+--------+



In [16]:
spark.sql("UPDATE local.vendas.clientes SET telefone = '(11) 91234-5678' WHERE id = 1")
spark.sql("UPDATE local.vendas.clientes SET telefone = '(21) 99876-5432' WHERE id = 2")
spark.sql("SELECT * FROM local.vendas.clientes").show()


+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
+---+---------------+------------------+--------------+------+---------------+



## MERGE — Upsert (Insert or Update)

Insere o registro se não existir, ou atualiza se já existir.

In [17]:
spark.sql("""
    MERGE INTO local.vendas.clientes AS target
    USING (
        SELECT 6 AS id, 'Pedro Alves' AS nome, 'pedro@email.com' AS email,
               'Fortaleza' AS cidade, 'CE' AS estado, '(85) 98765-4321' AS telefone
    ) AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM local.vendas.clientes").show()


+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
|  6|    Pedro Alves|   pedro@email.com|     Fortaleza|    CE|(85) 98765-4321|
+---+---------------+------------------+--------------+------+---------------+



## Snapshots — Time Travel no Iceberg

O Iceberg usa **snapshots** para versionar o estado da tabela.

In [18]:
spark.sql("SELECT * FROM local.vendas.clientes.snapshots").show(truncate=False)


+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                      |summary                                                                             

In [19]:
snapshots = spark.sql(
    "SELECT snapshot_id, committed_at, operation "
    "FROM local.vendas.clientes.snapshots ORDER BY committed_at"
)
snapshots.show(truncate=False)

# Pega o snapshot_id mais antigo (apos o primeiro INSERT)
primeiro_snapshot = snapshots.first()['snapshot_id']
print(f'Primeiro snapshot_id: {primeiro_snapshot}')

# Time Travel — le tabela no estado do primeiro snapshot
df_snap = (
    spark.read
    .option('snapshot-id', primeiro_snapshot)
    .table('local.vendas.clientes')
)
print('Clientes no primeiro snapshot (so INSERT):')
df_snap.show()


+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|758366427453582650 |2026-05-07 00:34:58.836|append   |
|1996665160676102175|2026-05-07 00:35:09.59 |overwrite|
|4964420600291818565|2026-05-07 00:35:10.111|overwrite|
|6208702612911869783|2026-05-07 00:35:11.508|append   |
+-------------------+-----------------------+---------+

Primeiro snapshot_id: 758366427453582650
Clientes no primeiro snapshot (so INSERT):
+---+---------------+------------------+--------------+------+--------+
| id|           nome|             email|        cidade|estado|telefone|
+---+---------------+------------------+--------------+------+--------+
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|    NULL|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|    NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|    NULL|
|  4|     Joao Costa|    joao@email

In [20]:
spark.stop()
print('Sessao Spark encerrada.')


Sessao Spark encerrada.
